# Wczytanie checkpointu i wizualizacja (`usage1`)

Ten notebook **nie trenuje** modelu — ładuje wagę z pliku `.ckpt` (zapis z `train1.ipynb`) i pokazuje **RGB | predykcja | ground truth** na zbiorze testowym, tak jak ostatnia komórka w `train1.ipynb`.

Uruchom komórki **po kolei od góry**. W pierwszej komórce ustaw `CONFIG` (zwłaszcza `checkpoint_path` i `data_root`) tak samo jak przy treningu, żeby `NUM_CLASSES` i podział train/val/test były spójne z modelem.

In [1]:
import glob
import os
import random

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset, random_split

import torchvision.models as models
import torchvision.transforms as T

# --- Konfiguracja (jak w train1.ipynb — dopasuj ścieżki do swojego środowiska) ---
# data_root: katalog z sekwencjami CARLA (np. 00/, 04/…), każda z podfolderami rgb/ i segmentation_raw/.
# Nie ustawiaj tu samego katalogu repo (../.) — wtedy build_class_map znajdzie maseki przez **/,
# ale CARLADataset szuka tylko bezpośrednich podfolderów o nazwie złożonej z cyfr.
CONFIG = {
    "data_root": "../CARLA_dataset",
    "max_images_per_sequence": None,
    "batch_size": 2,
    "num_workers": 0,
    "train_frac": 0.7,
    "val_frac": 0.15,
    "checkpoint_path": "resnet_segmentation_train1.ckpt",
    "seed": 42,
}

random.seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CONFIG["seed"])


In [2]:
def build_class_map(root: str):
    """Zbiera unikalne ID klas z kanału R masek w segmentation_raw."""
    classes = set()
    pattern = os.path.join(root, "**", "segmentation_raw", "*.png")
    for mask_path in glob.glob(pattern, recursive=True):
        mask = np.array(Image.open(mask_path))[:, :, 0]
        classes.update(np.unique(mask).tolist())

    classes = sorted(classes)
    class_map = {int(v): i for i, v in enumerate(classes)}
    print("Znalezione klasy (raw):", classes)
    print("NUM_CLASSES =", len(classes))
    return class_map, len(classes)


def build_label_lut(class_map: dict, size: int = 256) -> np.ndarray:
    """Szybkie mapowanie uint8 -> indeks klasy (nieznane -> 0)."""
    lut = np.zeros(size, dtype=np.int64)
    for raw, idx in class_map.items():
        if 0 <= int(raw) < size:
            lut[int(raw)] = int(idx)
    return lut


class_map, NUM_CLASSES = build_class_map(CONFIG["data_root"])
LABEL_LUT = build_label_lut(class_map)


Znalezione klasy (raw): [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 18, 19, 20, 21, 22, 23, 24, 25, 27]
NUM_CLASSES = 25


In [3]:
class CARLADataset(Dataset):
    def __init__(self, root: str, label_lut: np.ndarray, max_per_sequence: int | None = None):
        self.samples = []
        self.label_lut = label_lut

        for folder in sorted(
            f for f in os.listdir(root)
            if os.path.isdir(os.path.join(root, f)) and f.isdigit()
        ):
            rgb_dir = os.path.join(root, folder, "rgb")
            mask_dir = os.path.join(root, folder, "segmentation_raw")
            if not os.path.isdir(rgb_dir) or not os.path.isdir(mask_dir):
                continue

            img_paths = sorted(glob.glob(os.path.join(rgb_dir, "*.png")))
            if max_per_sequence is not None:
                img_paths = img_paths[:max_per_sequence]

            for img_path in img_paths:
                name = os.path.basename(img_path)
                mask_path = os.path.join(mask_dir, name)
                if os.path.exists(mask_path):
                    self.samples.append((img_path, mask_path))

        self.img_tf = T.Compose([T.ToTensor()])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, mask_path = self.samples[idx]
        img = Image.open(img_path).convert("RGB")
        mask = np.array(Image.open(mask_path))[:, :, 0]
        mask = self.label_lut[mask.astype(np.int64)]
        mask = torch.from_numpy(mask).long()
        img = self.img_tf(img)
        return img, mask


In [4]:
class SegModel(nn.Module):
    def __init__(self, num_classes: int):
        super().__init__()
        resnet = models.resnet34(weights=models.ResNet34_Weights.DEFAULT)
        self.encoder = nn.Sequential(
            resnet.conv1,
            resnet.bn1,
            resnet.relu,
            resnet.maxpool,
            resnet.layer1,
            resnet.layer2,
            resnet.layer3,
            resnet.layer4,
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(512, 256, 2, stride=2),
            nn.ReLU(),
            nn.ConvTranspose2d(256, 128, 2, stride=2),
            nn.ReLU(),
            nn.ConvTranspose2d(128, 64, 2, stride=2),
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 2, stride=2),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 16, 2, stride=2),
            nn.ReLU(),
            nn.Conv2d(16, num_classes, 1),
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))


In [5]:
# Wizualizacja działania modelu (RGB | predykcja | GT)
# Wymaga: CONFIG, NUM_CLASSES, LABEL_LUT, SegModel, F, DataLoader, random_split, CARLADataset
# Checkpoint z komórki treningowej train1.ipynb lub sam plik .ckpt.

import matplotlib.pyplot as plt

_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

seg_demo = SegModel(num_classes=NUM_CLASSES).to(_device)
_ckpt = torch.load(CONFIG["checkpoint_path"], map_location=_device)
_sd = {k.replace("model.", "", 1): v for k, v in _ckpt["state_dict"].items() if k.startswith("model.")}
seg_demo.load_state_dict(_sd, strict=True)
seg_demo.eval()

try:
    _loader = test_loader
except NameError:
    _full = CARLADataset(
        CONFIG["data_root"],
        LABEL_LUT,
        max_per_sequence=CONFIG["max_images_per_sequence"],
    )
    _n = len(_full)
    _tn = int(CONFIG["train_frac"] * _n)
    _vn = int(CONFIG["val_frac"] * _n)
    _test_n = _n - _tn - _vn
    _, _, _test_sub = random_split(
        _full,
        [_tn, _vn, _test_n],
        generator=torch.Generator().manual_seed(CONFIG["seed"]),
    )
    _loader = DataLoader(_test_sub, batch_size=2, shuffle=False, num_workers=0)


def _idx_to_rgb(mask_hw, n_classes):
    lut = (plt.cm.turbo(np.linspace(0, 0.92, n_classes))[:, :3]).astype(np.float32)
    m = np.clip(mask_hw.detach().cpu().numpy(), 0, n_classes - 1)
    return lut[m]


N_SHOW = 10
_shown = 0

with torch.no_grad():
    for _imgs, _masks in _loader:
        _imgs = _imgs.to(_device)
        _logits = seg_demo(_imgs)
        _logits = F.interpolate(
            _logits,
            size=_masks.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )
        _pred = _logits.argmax(dim=1)
        for _b in range(_imgs.size(0)):
            if _shown >= N_SHOW:
                break
            _shown += 1
            _rgb = _imgs[_b].cpu().permute(1, 2, 0).numpy()
            _rgb = np.clip(_rgb, 0.0, 1.0)
            _fig, _ax = plt.subplots(1, 3, figsize=(14, 4))
            _ax[0].imshow(_rgb)
            _ax[0].set_title("Wejście (RGB)")
            _ax[0].axis("off")
            _ax[1].imshow(_idx_to_rgb(_pred[_b], NUM_CLASSES))
            _ax[1].set_title("Predykcja (argmax)")
            _ax[1].axis("off")
            _ax[2].imshow(_idx_to_rgb(_masks[_b], NUM_CLASSES))
            _ax[2].set_title("Ground truth")
            _ax[2].axis("off")
            _fig.suptitle(f"Przykład {_shown} / {N_SHOW} — zbiór testowy")
            plt.tight_layout()
            plt.show()
        if _shown >= N_SHOW:
            break

if _shown < N_SHOW:
    print(f"Uwaga: w zbiorze testowym jest tylko {_shown} próbek (pytano o {N_SHOW}).")


Uwaga: w zbiorze testowym jest tylko 0 próbek (pytano o 10).
